In [ ]:
# (Setup - you already have this in your notebook, skip it)
# In your notebook, df is the dataframe after the missing values section
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_excel("Project_Data.xlsx", keep_default_na=False, na_values=[""])

## 1.4 Outliers

### 1.4.1 Identifying Outliers

In [ ]:
num_cols = ["Age", "Consultation Duration", "Satisfaction Level",
            "Approximate Annual Income", "Consultation Price", "Insurance Coverage"]

# min and max already show some strange values (age 195, rating 6, price 39999...)
display(df[num_cols].describe().round(1))

# IQR rule (1.5 x IQR) for each variable
print()
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(col, "| rows outside the IQR limits:", n_out)

df[num_cols].plot(kind="box", subplots=True, layout=(2, 3), figsize=(14, 8))
plt.suptitle("Boxplots of the numerical variables (before treating outliers)")
plt.tight_layout()
plt.show()

The describe table and the boxplots show a few values that are far away from the rest: ages up to 195, prices up to £39,999, incomes up to £12M, durations up to 600 minutes and a rating of 6.

The IQR rule alone is not good to decide what to change. It marks about 2,300 incomes and 241 prices as outliers, but most of them are normal values: many children and students have an income of 0, which makes the distribution very skewed, and the price depends on the department (see below). So we use the IQR as a first look and then check each variable with what is possible in the real world and with the variable description in the project statement.

In [ ]:
# Age: the highest values
print("Highest ages:", np.sort(df["Age"].dropna().unique())[-6:])
print("Rows with age above 110:", (df["Age"] > 110).sum(), "| patients:", df.loc[df["Age"] > 110, "Patient ID"].nunique())
print("\nPatient 1198 ages:")
print(df.loc[df["Patient ID"] == 1198, "Age"].value_counts())
print()
display(df[df["Age"] > 110].groupby("Patient ID")[["Age", "Profession", "Education Level"]].first())

In [ ]:
# Consultation Duration: the highest values
print("Highest durations:", np.sort(df["Consultation Duration"].unique())[-12:])
print("Rows with duration above 120:", (df["Consultation Duration"] > 120).sum())

# Consultation Price: normal range by department (price <= 1000)
print("\nNormal price range by department:")
display(df[df["Consultation Price"] <= 1000].groupby("Department")["Consultation Price"].agg(["min", "max"]).round(0).T)

print("\nPrices above 1000:")
display(df.loc[df["Consultation Price"] > 1000, ["Patient ID", "Department", "Consultation Price"]].round(0))

In [ ]:
# Approximate Annual Income
high_income = df["Approximate Annual Income"] >= 1_000_000
print("Rows with income above 1 million:", high_income.sum(), "| patients:", df.loc[high_income, "Patient ID"].nunique())
print("Highest income in the other rows:", df.loc[~high_income, "Approximate Annual Income"].max())
print("Incomes above 1 million:", df.loc[high_income, "Approximate Annual Income"].round(0).sort_values().tolist())

# one example patient: normal visits vs extreme visits
print("\nPatient 1416 incomes:")
print(df.loc[df["Patient ID"] == 1416, "Approximate Annual Income"].round(0).sort_values().tolist())

# zero income
zero = df["Approximate Annual Income"] == 0
print("\nRows with income 0:", zero.sum(), "| share under 18 years old:", (df.loc[zero, "Age"] < 18).mean().round(2))

In [ ]:
# Satisfaction Level (valid values are 1 to 5)
print(df["Satisfaction Level"].value_counts().sort_index())
print("\nDepartments of the rating 6:")
print(df.loc[df["Satisfaction Level"] == 6, "Department"].value_counts())

In [ ]:
# Insurance Coverage: the insurance cannot pay more than the price of the consultation
print("Rows where coverage is higher than the price:", (df["Insurance Coverage"] > df["Consultation Price"]).sum())
display(df.loc[df["Insurance Coverage"] > df["Consultation Price"],
             ["Patient ID", "Insurance Provider", "Consultation Price", "Insurance Coverage"]].round(1))

**What we see**
- **Age:** only 5 patients have ages above 110 (118, 138, 142, 158 and 195), while all the others are 100 or less. Patient 1198 has 138 in 21 visits and 38 in one visit, so it looks like a typing error with an extra "1" in front. Subtracting 100 gives ages that fit the profession and education of each patient (for example 18 for a high school student and 95 for a retired patient).
- **Consultation Duration:** all normal visits last between 15 and 120 minutes. After 120 the next value is 228, and 10 rows are above 120.
- **Consultation Price:** every normal price is between £50 and £400, and each department has its own range (for example ENT £50-100 and Emergency £200-400). That is why the IQR flags so many normal prices. 7 prices are around £25,000-£40,000, which is about 100 times too high: dividing by 100 gives £253-£400.
- **Income:** 21 rows (10 patients) are above £1M, while the highest income in all the other rows is £113,120. 16 of them are around £1.1M and 5 are around £11M-£12M. Dividing the first group by 10 and the second by 100 gives £113k-£120k, which is close to the highest normal incomes. Patient 1416 shows it well: the normal visits are between £54k and £108k, so £11k would be too low for this patient. Income 0 is not an outlier, because 91% of these rows are patients under 18.
- **Satisfaction Level:** the scale goes from 1 to 5, but 719 rows have a 6. The IQR does not find it because 6 is close to the other values. 99% of the 6s are in only two departments (ENT and General Practice), so it looks like a problem in the form of these departments and not a real rating.
- **Insurance Coverage:** 6 rows have a coverage higher than the price, which is impossible. The IQR flags 77 coverages, but most of them are normal values from expensive consultations.

### 1.4.2 Handling Outliers

| Variable | Outlier | Decision | Why |
|---|---|---|---|
| Age | 103 rows (5 patients) above 110 | Correct: subtract 100 | Typing error with an extra "1", confirmed by patient 1198 and by the profession and education |
| Consultation Price | 7 rows above £1000 | Correct: divide by 100 | The result is inside the normal range of the department |
| Approximate Annual Income | 21 rows above £1M | Correct: divide by 10 (£1M-£10M) or by 100 (above £10M) | The result is close to the highest normal incomes and to the patient's other visits |
| Income = 0 | 110 patients | Keep | Valid value (mostly children and students) |
| Consultation Duration | 10 rows above 120 min | Impute with the median of the department | The extreme values are not a clear multiple of a normal duration, so the real value is unknown |
| Satisfaction Level | 719 rows with a 6 | Set to missing | The rating is impossible and the real opinion is unknown. We do not change it to 5 because it would invent good ratings |
| Insurance Coverage | 6 rows higher than the price | Recompute: provider rate x price | All the other rows follow this rule exactly |

We do not remove any row or patient: the errors have a clear pattern or can be replaced, and removing visits would lose real information (and patients).

In [ ]:
# Age: extra "1" in front -> subtract 100
df.loc[df["Age"] > 110, "Age"] = df.loc[df["Age"] > 110, "Age"] - 100

# Price: 100 times too high -> divide by 100
df.loc[df["Consultation Price"] > 1000, "Consultation Price"] = df.loc[df["Consultation Price"] > 1000, "Consultation Price"] / 100

# Income: 100 times too high (above 10 million) or 10 times too high (above 1 million)
df.loc[df["Approximate Annual Income"] >= 10_000_000, "Approximate Annual Income"] = df.loc[df["Approximate Annual Income"] >= 10_000_000, "Approximate Annual Income"] / 100
df.loc[df["Approximate Annual Income"] >= 1_000_000, "Approximate Annual Income"] = df.loc[df["Approximate Annual Income"] >= 1_000_000, "Approximate Annual Income"] / 10

# Duration: above 120 -> missing -> median of the department
df.loc[df["Consultation Duration"] > 120, "Consultation Duration"] = np.nan
df["Consultation Duration"] = df["Consultation Duration"].fillna(df.groupby("Department")["Consultation Duration"].transform("median"))

# Satisfaction: 6 is not a valid rating -> missing
df.loc[df["Satisfaction Level"] == 6, "Satisfaction Level"] = np.nan

# Coverage: higher than the price -> provider rate x price
rate_by_provider = {"None": 0.0, "Provider A": 0.8, "Provider B": 0.9, "Provider C": 0.7, "Provider D": 1.0}
too_high = df["Insurance Coverage"] > df["Consultation Price"]
df.loc[too_high, "Insurance Coverage"] = df.loc[too_high, "Insurance Provider"].map(rate_by_provider) * df.loc[too_high, "Consultation Price"]

# check the result
display(df[num_cols].describe().round(1))
print("\nAges above 110:", (df["Age"] > 110).sum())
print("Prices above 1000:", (df["Consultation Price"] > 1000).sum())
print("Incomes above 1 million:", (df["Approximate Annual Income"] >= 1_000_000).sum())
print("Durations above 120:", (df["Consultation Duration"] > 120).sum())
print("Coverage higher than the price:", (df["Insurance Coverage"] > df["Consultation Price"]).sum())
print("Missing ratings (the old 6s):", df["Satisfaction Level"].isna().sum())
print("Rows and patients:", len(df), "|", df["Patient ID"].nunique())

df[num_cols].plot(kind="box", subplots=True, layout=(2, 3), figsize=(14, 8))
plt.suptitle("Boxplots of the numerical variables (after treating outliers)")
plt.tight_layout()
plt.show()

After the corrections all the values are inside the possible ranges and we still have the 10,008 visits of the 500 patients. The boxplots now show compact distributions without the extreme points.

Note for the ABT: the 719 ratings that became missing are ignored by `mean()` and `min()`. The share of low ratings should be calculated only with the valid ratings (for example with `dropna()` first).